## 2.2 Tokenizing text

In [29]:
import os
import urllib.request

# Download a small text file once so we have some real data to tokenize.
# Keeping it local avoids re-downloading on every run.
if not os.path.exists('the-verdict.txt'):
    url = 'https://raw.githubusercontent.com/GlebTanaka/LLMs-from-scratch/refs/heads/main/ch02/01_main-chapter-code/the-verdict.txt'
    file_path = 'the-verdict.txt'
    urllib.request.urlretrieve(url, file_path)

In [30]:
# Read the file and get character count (raw characters, not tokens).
# Character count shows the raw size of the text before any tokenization.
with open('the-verdict.txt', 'r', encoding='utf-8') as file:
    the_verdict_text = file.read()
    char_count = len(the_verdict_text)
print(f"Total number of characters in the file: {char_count}")
print(the_verdict_text[:99])

Total number of characters in the file: 20479
I HAD always thought Jack Gisburn rather a cheap genius--though a good fellow enough--so it was no 


In [31]:
import re

# A simple sentence to illustrate regex-based tokenization.
sample_sentence = "Hello, world! This is a simple example of tokenization (using regex)."

# Tokenize using re.findall
# \w+ matches one or more word characters (letters, digits, underscores)
tokens = re.findall(r'\w+', sample_sentence.lower())

print("Original text:", sample_sentence)
print("\nTokens:", tokens)
print("Number of tokens:", len(tokens))

Original text: Hello, world! This is a simple example of tokenization (using regex).

Tokens: ['hello', 'world', 'this', 'is', 'a', 'simple', 'example', 'of', 'tokenization', 'using', 'regex']
Number of tokens: 11


In [32]:
# A few example texts with different patterns
texts = [
    "Hello world",                    # Simple space-separated
    "Hello, world!",                  # With punctuation
    "Is this--a test?",              # With double dash
    "Word. Another word... Final"     # With multiple dots
]

# Try different patterns
patterns = [
    r'\w+',                           # Only word characters (drops punctuation)
    r'[A-Za-z]+',                     # Letters only (numbers and underscores dropped)
    r'([,.:;?_!"()\']|--|\s)',        # Capturing punctuation and whitespace as separators
    r'\S+'                            # Greedy chunks of non-whitespace (keeps punctuation attached)
]

# Demonstrate how each pattern behaves on each text.
# Important: When using re.split with a capturing group, the separators are included
# in the result; we then strip/filter them below if desired.
for text in texts:
    print(f"\nOriginal text: '{text}'")
    for pattern in patterns:
        print(f"\nPattern '{pattern}':")
        if pattern.startswith('('):
            # If pattern has a capturing group, use split so we can preserve punctuation as separate tokens.
            tokens = [t.strip() for t in re.split(pattern, text) if t.strip()]
        else:
            # Otherwise, find all matching tokens (punctuation will be dropped in these cases).
            tokens = re.findall(pattern, text)
        print(f"Tokens: {tokens}")


Original text: 'Hello world'

Pattern '\w+':
Tokens: ['Hello', 'world']

Pattern '[A-Za-z]+':
Tokens: ['Hello', 'world']

Pattern '([,.:;?_!"()\']|--|\s)':
Tokens: ['Hello', 'world']

Pattern '\S+':
Tokens: ['Hello', 'world']

Original text: 'Hello, world!'

Pattern '\w+':
Tokens: ['Hello', 'world']

Pattern '[A-Za-z]+':
Tokens: ['Hello', 'world']

Pattern '([,.:;?_!"()\']|--|\s)':
Tokens: ['Hello', ',', 'world', '!']

Pattern '\S+':
Tokens: ['Hello,', 'world!']

Original text: 'Is this--a test?'

Pattern '\w+':
Tokens: ['Is', 'this', 'a', 'test']

Pattern '[A-Za-z]+':
Tokens: ['Is', 'this', 'a', 'test']

Pattern '([,.:;?_!"()\']|--|\s)':
Tokens: ['Is', 'this', '--', 'a', 'test', '?']

Pattern '\S+':
Tokens: ['Is', 'this--a', 'test?']

Original text: 'Word. Another word... Final'

Pattern '\w+':
Tokens: ['Word', 'Another', 'word', 'Final']

Pattern '[A-Za-z]+':
Tokens: ['Word', 'Another', 'word', 'Final']

Pattern '([,.:;?_!"()\']|--|\s)':
Tokens: ['Word', '.', 'Another', 'word', '.',

In [33]:
# Define a splitter that keeps punctuation as separate tokens and splits on whitespace.
# This helps later when we want a consistent reversible decode.
pattern = r'([,.:;?_!"()\']|--|\s)'

# Tokenize the full text using the pattern above.
# We strip out empty strings and pure separators, but punctuation tokens are kept.
tokens = [token.strip() for token in re.split(pattern, the_verdict_text) if token.strip()]

print("\nTokens:", tokens[:30])  # Preview the first 30 tokens
print("Number of tokens:", len(tokens))



Tokens: ['I', 'HAD', 'always', 'thought', 'Jack', 'Gisburn', 'rather', 'a', 'cheap', 'genius', '--', 'though', 'a', 'good', 'fellow', 'enough', '--', 'so', 'it', 'was', 'no', 'great', 'surprise', 'to', 'me', 'to', 'hear', 'that', ',', 'in']
Number of tokens: 4690


## 2.3 Converting tokens into token IDs

In [34]:
# Build a vocabulary: the set of unique tokens we saw in the text.
# Sorting is not required, but makes the mapping deterministic/reproducible.
vocab = sorted(set(tokens))

print("Vocabulary (unique tokens):", vocab[:5])
print("Vocabulary size:", len(vocab))

# Create a token->id mapping (string to integer).
# Each distinct token gets a unique index.
token_to_id = {token: idx for idx, token in enumerate(vocab)}

# Convert the text tokens into their corresponding IDs.
token_ids = [token_to_id[token] for token in tokens]

# Print first 5 items from token_to_id dictionary
print("\nFirst 5 token-to-ID mappings:")
for i, (token, id_num) in enumerate(token_to_id.items()):
    if i >= 5:
        break
    print(f"'{token}' -> {id_num}")

# Print first 10 token IDs
print("\nFirst 10 tokens as IDs:", token_ids[:10])

Vocabulary (unique tokens): ['!', '"', "'", '(', ')']
Vocabulary size: 1130

First 5 token-to-ID mappings:
'!' -> 0
'"' -> 1
''' -> 2
'(' -> 3
')' -> 4

First 10 tokens as IDs: [53, 44, 149, 1003, 57, 38, 818, 115, 256, 486]


In [35]:
class SimpleTokenizerV1:
    # Minimal tokenizer that assumes every token exists in the given vocabulary.
    # If you pass unseen tokens, it will raise a KeyError.
    def __init__(self, vocab):
        self.str_to_int = vocab # Dictionary mapping tokens to IDs
        self.int_to_str = {i:s for s,i in vocab.items()} # Reverse mapping: IDs to tokens

    def encode(self, text):
        # Split text into tokens using the same pattern as above so encode/decode are consistent.
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)
        # Remove empty strings and whitespace-only items; keep punctuation tokens.
        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Map tokens to IDs (assumes all tokens exist in vocab)
        ids = [self.str_to_int[s] for s in preprocessed]
        return ids

    def decode(self, ids):
        # Inverse mapping: join tokens with spaces so words are separated.
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [36]:
# Example usage of SimpleTokenizerV1
vocab = {'hello': 0, 'world': 1, '!': 2}
tokenizer = SimpleTokenizerV1(vocab)

# Encoding: turns text into a sequence of IDs
ids = tokenizer.encode("hello world!")  # Returns: [0, 1, 2]

# Decoding: turns IDs back into text (should be reversible for known tokens)
text = tokenizer.decode([0, 1, 2])  # Returns: "hello world!"
print(text)

hello world!


In [37]:
# Using tokenizer on prior created vocabulary.
tokenizer_2 = SimpleTokenizerV1(token_to_id)

text_section = """"It's the last he painted, you know,"
           Mrs. Gisburn said with pardonable pride."""
# Note: If the text contains tokens not in token_to_id, this would error.
# Here it should work because token_to_id was built from the same file.
ids = tokenizer_2.encode(text_section)
print(ids)

[1, 56, 2, 850, 988, 602, 533, 746, 5, 1126, 596, 5, 1, 67, 7, 38, 851, 1108, 754, 793, 7]


In [38]:
# decode integers into text
tokenizer_2.decode(ids)

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

In [39]:
# First tokenizer_2.encode(text_section) converts the text into a list of token IDs
# Then tokenizer_2.decode() converts those IDs back into text
# This is a roundtrip conversion: text -> IDs -> text
# Useful for testing if the tokenizer preserves the text correctly
tokenizer_2.decode(tokenizer_2.encode(text_section))

'" It\' s the last he painted, you know," Mrs. Gisburn said with pardonable pride.'

## 2.4 Adding special context tokens

In [40]:
# Check length before adding special tokens
len(token_to_id.items())

1130

In [41]:
# Add special tokens - using the next available IDs
next_id = len(token_to_id)  # Get the next available ID

# Special token commonly used to mark an explicit boundary/end of a sample
token_to_id['<|endoftext|>'] = next_id
next_id += 1

# Special token for unknown/out-of-vocabulary tokens (robustness)
token_to_id[('<|unk|>')] = next_id

# Print the new tokens and their IDs
print("Special tokens added:")
print(f"<|endoftext|> token ID: {token_to_id['<|endoftext|>']}")
print(f"<|unk|> token ID: {token_to_id['<|unk|>']}")

Special tokens added:
<|endoftext|> token ID: 1130
<|unk|> token ID: 1131


In [42]:
# length after adding special tokens:
len(token_to_id.items())

1132

In [43]:
# Peek at the last few entries to confirm insertion
for i, item in enumerate(list(token_to_id.items())[-5:]):
    print(item)

('younger', 1127)
('your', 1128)
('yourself', 1129)
('<|endoftext|>', 1130)
('<|unk|>', 1131)


In [44]:
# Improved tokenizer that safely handles unknown tokens by mapping them to <|unk|>
class SimpleTokenizerV2:
    def __init__(self, vocab):
        self.str_to_int = vocab
        self.int_to_str = {i:s for s,i in vocab.items()}
        # Store the '<|unk|>' token ID for easy access
        self.unk_token_id = vocab['<|unk|>'] # cached for speed

    def encode(self, text):
        preprocessed = re.split(r'([,.:;?_!"()\']|--|\s)', text)

        preprocessed = [
            item.strip() for item in preprocessed if item.strip()
        ]
        # Use get() with unk_token_id as default for unknown words
        ids = [self.str_to_int.get(s, self.unk_token_id) for s in preprocessed]
        return ids

    def decode(self, ids):
        text = " ".join([self.int_to_str[i] for i in ids])
        # Replace spaces before the specified punctuations
        text = re.sub(r'\s+([,.?!"()\'])', r'\1', text)
        return text

In [45]:
tokenizer = SimpleTokenizerV2(token_to_id)
# This will work even with words not in our vocabulary
test_text = "hello nonexistentword world"
encoded = tokenizer.encode(test_text)
decoded = tokenizer.decode(encoded)
print(decoded)

<|unk|> <|unk|> <|unk|>


In [46]:
# Example showing how a special end-of-text token can be embedded into a sequence
text1 = "Hello, do you like tea?"
text2 = "In the sunlit terraces of the palace."

# Join with the special boundary token; some training pipelines use this to separate samples
text = " <|endoftext|> ".join((text1, text2))

print(text)

Hello, do you like tea? <|endoftext|> In the sunlit terraces of the palace.


In [47]:
# Encoding shows the special token appears as its own ID
tokenizer.encode(text)

[1131, 5, 355, 1126, 628, 975, 10, 1130, 55, 988, 956, 984, 722, 988, 1131, 7]

In [48]:
# Decoding reconstructs the string and removes extra spaces before punctuation
tokenizer.decode(tokenizer.encode(text))

'<|unk|>, do you like tea? <|endoftext|> In the sunlit terraces of the <|unk|>.'

## 2.5 Byte pair encoding

In [49]:
import tiktoken
tiktoken.__version__

'0.9.0'

In [50]:
import tiktoken

# tiktoken provides fast production-grade tokenizers (e.g., GPT-2 BPE).
# Here we instantiate the GPT-2 tokenizer by name.
encoding = tiktoken.get_encoding("gpt2")

# Example text
text = "Hello world! This is an example of tiktoken tokenization."

# Encode text to integer token IDs (model-specific vocabulary)
tokens = encoding.encode(text)
print("Encoded tokens:", tokens)

# Decode IDs back to text; should be lossless for supported characters
decoded_text = encoding.decode(tokens)
print("\nDecoded text:", decoded_text)

# Get the number of tokens
print(f"\nNumber of tokens: {len(tokens)}")

# Inspect each token to see its string form; BPE can split inside words
print("\nToken-by-token breakdown:")
for token in tokens:
    print(f"Token {token}: {encoding.decode([token])!r}")

Encoded tokens: [15496, 995, 0, 770, 318, 281, 1672, 286, 256, 1134, 30001, 11241, 1634, 13]

Decoded text: Hello world! This is an example of tiktoken tokenization.

Number of tokens: 14

Token-by-token breakdown:
Token 15496: 'Hello'
Token 995: ' world'
Token 0: '!'
Token 770: ' This'
Token 318: ' is'
Token 281: ' an'
Token 1672: ' example'
Token 286: ' of'
Token 256: ' t'
Token 1134: 'ik'
Token 30001: 'token'
Token 11241: ' token'
Token 1634: 'ization'
Token 13: '.'


In [51]:
# List available encodings and compare tokenizations across them.
print("Available encodings:", tiktoken.list_encoding_names())

# Create encoders for different models
gpt2_encoder = tiktoken.get_encoding("gpt2")
cl100k_encoder = tiktoken.get_encoding("cl100k_base")  # Used by GPT-4

# Compare tokenization
text = "Hello world! Let's try different encoders."
print("\nGPT-2 tokens:", gpt2_encoder.encode(text))
print("CL100K tokens:", cl100k_encoder.encode(text))

Available encodings: ['gpt2', 'r50k_base', 'p50k_base', 'p50k_edit', 'cl100k_base', 'o200k_base']

GPT-2 tokens: [15496, 995, 0, 3914, 338, 1949, 1180, 2207, 375, 364, 13]
CL100K tokens: [9906, 1917, 0, 6914, 596, 1456, 2204, 3289, 53598, 13]


## 2.6 Data sampling with sliding window

The code demonstrates sliding window sampling by:
- Taking a section of encoded text
- Creating input-target pairs
- Showing how context can be built up progressively
- Demonstrating both the numerical (token IDs) and text representations

This helps in understanding how language models process text as a sequence of tokens and how they use context to predict the next token

In [52]:
# Read the input text file
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

# Convert the raw text into token IDs using our tokenizer
enc_text = tokenizer.encode(raw_text)
print(f"Total number of tokens in the text: {len(enc_text)}")

# Take a sample starting from the 50th token to work with
enc_sample = enc_text[50:]
# Define the size of our context window
context_size = 4

# Create input (x) and target (y) sequences for our first example
# x contains tokens [0:4] and y contains tokens [1:5]
# y is x shifted by one position — this is next-token prediction.
x = enc_sample[:context_size]
y = enc_sample[1:context_size+1]

print("\nDemonstrating basic input-target pair:")
print(f"Input sequence (x):  {x}")
print(f"Target sequence (y): {y}")

print("\nDemonstrating progressive context building with token IDs:")
# Show how we can build up context one token at a time
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # Print the increasing context size and the next token to predict
    print(f"Context {i} tokens: {context} ----> Next token: {desired}")

print("\nDemonstrating progressive context building with decoded text:")
# Same as above but show the actual text instead of token IDs
for i in range(1, context_size+1):
    context = enc_sample[:i]
    desired = enc_sample[i]
    # Print the increasing context size and the next token to predict, but in readable text
    print(f"Context: '{tokenizer.decode(context)}' ----> Next token: '{tokenizer.decode([desired])}'")

Total number of tokens in the text: 4690

Demonstrating basic input-target pair:
Input sequence (x):  [568, 115, 1066, 727]
Target sequence (y): [115, 1066, 727, 988]

Demonstrating progressive context building with token IDs:
Context 1 tokens: [568] ----> Next token: 115
Context 2 tokens: [568, 115] ----> Next token: 1066
Context 3 tokens: [568, 115, 1066] ----> Next token: 727
Context 4 tokens: [568, 115, 1066, 727] ----> Next token: 988

Demonstrating progressive context building with decoded text:
Context: 'in' ----> Next token: 'a'
Context: 'in a' ----> Next token: 'villa'
Context: 'in a villa' ----> Next token: 'on'
Context: 'in a villa on' ----> Next token: 'the'


In [53]:
import torch
print("PyTorch version:", torch.__version__)

PyTorch version: 2.7.1


In [54]:
from torch.utils.data import Dataset, DataLoader

class GPTDatasetV1(Dataset):
    """
    Creates overlapping input/target pairs from a long text using a sliding window.
    Each input is a sequence of length `max_length`; the target is the same sequence shifted by 1.
    """
    def __init__(self, txt, tokenizer, max_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenize the entire text
        token_ids = tokenizer.encode(txt, allowed_special={"<|endoftext|>"})
        # We need at least max_length+1 tokens to form one (input, target) pair
        assert len(token_ids) > max_length, "Number of tokenized inputs must at least be equal to max_length+1"

        # Sliding window: step forward by `stride` tokens each time, creating overlapping chunks.
        # Overlap helps the model see similar contexts starting at different positions.
        for i in range(0, len(token_ids) - max_length, stride):
            input_chunk = token_ids[i:i + max_length]
            target_chunk = token_ids[i + 1: i + max_length + 1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        # Returns a pair: (input sequence, next-token targets)
        return self.input_ids[idx], self.target_ids[idx]

In [55]:
def create_dataloader_v1(txt, batch_size=4, max_length=256,
                         stride=128, shuffle=True, drop_last=True,
                         num_workers=0):
    """
    Convenience helper to build a PyTorch DataLoader for next-token prediction.
    Uses a fast production tokenizer (GPT-2 BPE via tiktoken) for consistency with common LLMs.
    """
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset (sliding-window sequences)
    dataset = GPTDatasetV1(txt, tokenizer, max_length, stride)

    # Wrap in a DataLoader to iterate in batches during training
    dataloader = DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=shuffle,     # shuffle improves training stability
        drop_last=drop_last, # ensures all batches have the same size
        num_workers=num_workers
    )

    return dataloader

In [56]:
# Load raw text again for data loader demo
with open("the-verdict.txt", "r", encoding="utf-8") as f:
    raw_text = f.read()

In [57]:
# Create a small dataloader to visualize shapes and batching behavior.
dataloader = create_dataloader_v1(
    raw_text,
    batch_size=1,      # Only one sequence per batch
    max_length=4,      # Each sequence is 4 tokens long
    stride=1,          # Move window by 1 token each time
    shuffle=False      # Keep sequences in order
)

data_iter = iter(dataloader)
first_batch = next(data_iter)
print(first_batch)

[tensor([[  40,  367, 2885, 1464]]), tensor([[ 367, 2885, 1464, 1807]])]


In [58]:
second_batch = next(data_iter)
print(second_batch)

[tensor([[ 367, 2885, 1464, 1807]]), tensor([[2885, 1464, 1807, 3619]])]


## 2.7 Creating Token embeddings

In [59]:
import torch.nn as nn

# Let's say we have a small vocabulary of 5 words
vocab_size = 5        # Total unique tokens in our vocabulary
embedding_dim = 3     # Size of the embedding vector for each token

# Create an embedding layer
embedding_layer = nn.Embedding(
    num_embeddings=vocab_size,  # Size of the vocabulary
    embedding_dim=embedding_dim  # Size of the embedding vector
)

# Example input: batch of token IDs
# Let's say we have 2 sequences, each with 4 tokens
token_ids = torch.tensor([
    [0, 2, 1, 3],  # First sequence
    [1, 1, 4, 2]   # Second sequence
])

# Get embeddings for these tokens
embedded_tokens = embedding_layer(token_ids)

print("Input shape:", token_ids.shape)
print("Output shape:", embedded_tokens.shape)
print("\nEmbeddings for first sequence:")
print(embedded_tokens[0])

Input shape: torch.Size([2, 4])
Output shape: torch.Size([2, 4, 3])

Embeddings for first sequence:
tensor([[ 0.6835, -0.6427, -0.2303],
        [-0.5818,  0.0146,  0.3716],
        [ 1.2453, -0.0212, -0.4327],
        [ 0.0868,  1.0452, -1.0809]], grad_fn=<SelectBackward0>)


## 2.8 Encoding word positions

In [60]:
import tiktoken

# Initialize tokenizer
tokenizer = tiktoken.get_encoding("gpt2")

# Example text
text = "Hello world!"

# Convert text to token IDs and add a batch dimension so the shape becomes [batch, seq_len]
# e.g., [[15496, 995, 0]] depending on the tokenizer
token_ids = tokenizer.encode(text)
token_ids = torch.tensor([token_ids])  # Add batch dimension

# Create embedding layer with GPT-2 vocabulary size
# - num_embeddings: vocabulary size (index upper bound is vocab_size-1)
# - embedding_dim: size of each token vector
# For demonstration we use GPT‑2's common sizes (50257 vocab, 768 hidden size)
gpt2_embedding = nn.Embedding(
    num_embeddings=50257,  # GPT-2 vocabulary size
    embedding_dim=768  # Standard GPT-2 embedding dimension
)

# Look up token embeddings => shape: [batch, seq_len, embedding_dim]
embeddings = gpt2_embedding(token_ids)

print("Token IDs shape:", token_ids.shape)      # e.g., [1, 3]
print("Embeddings shape:", embeddings.shape)    # e.g., [1, 3, 768]

Token IDs shape: torch.Size([1, 3])
Embeddings shape: torch.Size([1, 3, 768])


### Example from the book: learnable token + position embeddings

In [61]:
# Token embedding configuration
vocab_size = 50257  # tokenizer vocab size
output_dim = 256    # model hidden size (embedding dimension)

# Learnable token embedding table: maps token IDs -> vectors in R^{output_dim}
token_embedding_layer = torch.nn.Embedding(vocab_size, output_dim)

# Suppose our training pipeline feeds batches of fixed-length sequences (max_length tokens each)
# We'll draw a single batch from the dataloader just to demonstrate shapes.
max_length = 4
dataloader = create_dataloader_v1(
    raw_text, batch_size=8, max_length=max_length, stride=max_length, shuffle=False
)
data_iter = iter(dataloader)
inputs, targets = next(data_iter)  # inputs: [batch, seq_len]
print("Token IDs:\n", inputs)
print("\nInputs shape:\n", inputs.shape) # e.g., torch.Size([8, 4])

Token IDs:
 tensor([[   40,   367,  2885,  1464],
        [ 1807,  3619,   402,   271],
        [10899,  2138,   257,  7026],
        [15632,   438,  2016,   257],
        [  922,  5891,  1576,   438],
        [  568,   340,   373,   645],
        [ 1049,  5975,   284,   502],
        [  284,  3285,   326,    11]])

Inputs shape:
 torch.Size([8, 4])


In [62]:
# Embed the tokens: result has shape [batch, seq_len, output_dim]
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)
print(token_embeddings)

torch.Size([8, 4, 256])
tensor([[[-0.8445, -0.1456, -0.4368,  ..., -0.7573,  2.0737,  0.0226],
         [ 1.0538, -0.3012,  0.0106,  ..., -0.8743,  0.0398, -0.7183],
         [-0.4510, -0.0469,  1.1020,  ..., -1.6693,  0.3270,  1.1077],
         [-1.6591, -1.4153, -1.9034,  ...,  0.2761, -1.0037,  0.2738]],

        [[-1.6869, -0.2683,  0.9940,  ...,  0.1692, -0.8604, -0.5278],
         [-0.1732, -0.8773, -0.6355,  ...,  0.1749,  0.5643, -1.6418],
         [ 0.8439, -1.4425, -0.5298,  ..., -0.1178, -0.1353,  0.5340],
         [ 0.4835,  1.1437, -2.1523,  ...,  1.1445,  3.2219, -0.4195]],

        [[ 1.2387,  1.5995, -1.1776,  ...,  1.1382, -0.2738,  0.1179],
         [ 0.4558,  0.8141, -0.6032,  ..., -1.0628,  1.3405, -1.1888],
         [ 0.5007, -1.0561, -0.9715,  ...,  0.9357, -0.7911, -0.4654],
         [ 0.7282,  0.4443,  1.1158,  ...,  0.5310,  1.9146, -1.6716]],

        ...,

        [[ 0.2856, -1.2702, -1.5599,  ..., -0.1473, -2.4766,  0.9396],
         [-0.7309,  0.9188, -0.23

In [63]:
# Create a learnable position embedding table:
# - We allocate one vector per absolute position in the context (0..context_length-1)
# - context_length must be >= the longest sequence length you feed the model
context_length = max_length
pos_embedding_layer = torch.nn.Embedding(context_length, output_dim)

# Inspect the randomly initialized position vectors (shape: [context_length, output_dim])
print("Raw position table (learnable weights):")
print(pos_embedding_layer.weight)

Raw position table (learnable weights):
Parameter containing:
tensor([[ 0.4116, -0.4047, -0.5176,  ...,  1.3941,  0.8295,  0.1922],
        [ 0.6262, -0.8924, -0.4086,  ..., -0.7401,  0.5870, -0.6242],
        [ 0.0997, -0.5865,  0.2501,  ..., -0.0884, -0.4717,  1.1210],
        [ 0.8796, -0.2113,  1.4866,  ...,  0.9401, -0.2436, -1.7336]],
       requires_grad=True)


In [64]:
# Build position indices for the sequence length [0, 1, 2, ..., seq_len-1]
# For a seq_len of 4, this is tensor([0, 1, 2, 3])
pos_indices = torch.arange(max_length)

# Lookup position embeddings => shape: [seq_len, output_dim]
pos_embeddings = pos_embedding_layer(pos_indices)
print("Position embeddings shape:", pos_embeddings.shape)  # e.g., [4, 256]
print(pos_embeddings)

Position embeddings shape: torch.Size([4, 256])
tensor([[ 0.4116, -0.4047, -0.5176,  ...,  1.3941,  0.8295,  0.1922],
        [ 0.6262, -0.8924, -0.4086,  ..., -0.7401,  0.5870, -0.6242],
        [ 0.0997, -0.5865,  0.2501,  ..., -0.0884, -0.4717,  1.1210],
        [ 0.8796, -0.2113,  1.4866,  ...,  0.9401, -0.2436, -1.7336]],
       grad_fn=<EmbeddingBackward0>)


In [65]:
# Combine token and position information:
# - token_embeddings shape: [batch, seq_len, output_dim]
# - pos_embeddings   shape: [seq_len, output_dim]
# PyTorch will broadcast pos_embeddings across the batch dimension automatically:
# [batch, seq_len, output_dim] + [seq_len, output_dim] -> [batch, seq_len, output_dim]
# If you prefer being explicit, you can unsqueeze: pos_embeddings[None, :, :] (shape [1, seq_len, output_dim])
input_embeddings = token_embeddings + pos_embeddings
print("Final input embeddings shape:", input_embeddings.shape)  # e.g., [8, 4, 256]
print(input_embeddings)

Final input embeddings shape: torch.Size([8, 4, 256])
tensor([[[-0.4329, -0.5503, -0.9544,  ...,  0.6369,  2.9032,  0.2148],
         [ 1.6800, -1.1936, -0.3981,  ..., -1.6145,  0.6268, -1.3425],
         [-0.3513, -0.6335,  1.3521,  ..., -1.7577, -0.1448,  2.2288],
         [-0.7794, -1.6266, -0.4168,  ...,  1.2162, -1.2473, -1.4598]],

        [[-1.2752, -0.6730,  0.4764,  ...,  1.5634, -0.0310, -0.3357],
         [ 0.4530, -1.7697, -1.0441,  ..., -0.5652,  1.1513, -2.2659],
         [ 0.9437, -2.0291, -0.2797,  ..., -0.2062, -0.6070,  1.6551],
         [ 1.3631,  0.9324, -0.6657,  ...,  2.0845,  2.9782, -2.1531]],

        [[ 1.6503,  1.1948, -1.6952,  ...,  2.5323,  0.5557,  0.3101],
         [ 1.0820, -0.0783, -1.0118,  ..., -1.8030,  1.9274, -1.8130],
         [ 0.6004, -1.6426, -0.7214,  ...,  0.8474, -1.2628,  0.6556],
         [ 1.6078,  0.2329,  2.6025,  ...,  1.4710,  1.6709, -3.4053]],

        ...,

        [[ 0.6972, -1.6749, -2.0774,  ...,  1.2468, -1.6471,  1.1318],
   

# Notes and tips:
- These are learned “absolute” position embeddings. They work up to context_length tokens.
If you later use longer sequences than context_length, you must enlarge the table and re-train or re-init.
- For variable-length batches with padding, ensure position indices only cover true tokens (not pad), or mask attention accordingly so pads don’t affect learning.
- Alternative schemes include sinusoidal or rotary embeddings; the add step stays the same: token + position.

